# World Triathlon Data Ingestion Pipeline: Mixed Relay OQR

このパイプラインは、大会(Event) -> 種目(Program) -> チームリザルト(Team Results) の階層構造を正確に取得し、分析しやすいフラットな形式に整理します。

## 1. Configuration
API設定と取得条件を外部変数として定義します。

In [ ]:
import requests
import pandas as pd
import time
import os
from pandas import json_normalize
from dotenv import load_dotenv

# .envファイルを読み込む
load_dotenv()

# --- 外部変数 (Externalized Variables) ---
CONFIG = {
    'API_KEY': os.getenv('WTRI_API_KEY'),
    'BASE_URL': 'https://api.triathlon.org/v1',
    'QUERY_PARAMS': {
        'specification_id': '380',               # 380: Mixed Relay
        'category_id': '348|624|351|352|349|341', # 大会カテゴリ
        'start_date': '2022-05-01',
        'end_date': '2024-05-31',
        'order': 'asc',
        'per_page': 100
    },
    'POLITE_DELAY': 0.5
}

HEADERS = {
    'accept': 'application/json',
    'apikey': CONFIG['API_KEY']
}

## 2. Step 1 & 2: Events and Programs
1. 対象大会のリストを取得
2. 各大会の詳細情報とプログラム（レース種目）を抽出

**Data Objects:**
- `df_target_races`: 大会詳細 (Key: `event_id`)
- `df_target_programs`: 種目リスト (Key: `prog_id`, FK: `event_id`)

In [2]:
def fetch_all_events(params):
    events = []
    page = 1
    total_pages = 1
    while page <= total_pages:
        params['page'] = page
        resp = requests.get(f"{CONFIG['BASE_URL']}/events", headers=HEADERS, params=params)
        if resp.status_code != 200: break
        data = resp.json()
        events.extend(data.get('data', []))
        total_pages = data.get('last_page', 1)
        page += 1
        time.sleep(0.3)
    return events

# 1. 大会リストの取得
raw_events = fetch_all_events(CONFIG['QUERY_PARAMS'].copy())
event_ids = [e['event_id'] for e in raw_events]

# 2. 各大会の詳細とプログラムを取得
detailed_events = []
all_programs = []

print(f"Processing {len(event_ids)} events...")
for idx, eid in enumerate(event_ids):
    resp = requests.get(f"{CONFIG['BASE_URL']}/events/{eid}", headers=HEADERS)
    if resp.status_code == 200:
        data = resp.json().get('data', {})
        detailed_events.append(data)
        
        # プログラムに親イベント情報を付与
        progs = data.get('programs', [])
        for p in progs:
            p['event_id'] = eid
            p['event_title'] = data.get('event_title')
            all_programs.append(p)
            
    if (idx + 1) % 5 == 0: print(f"Fetched {idx+1}/{len(event_ids)} details")
    time.sleep(CONFIG['POLITE_DELAY'])

# 正規化して DataFrame 化
# --- カテゴリ抽出ロジック ---
def pick_priority_category(categories):
    if not categories or not isinstance(categories, list):
        return {'cat_id': None, 'cat_name': None}
    
    priority_ids = [624, 351, 348, 349, 340]
    # cat_id が文字列の場合もあるので数値に変換して比較
    cat_dict = {int(c.get('cat_id')): c for c in categories if c.get('cat_id')}
    
    for pid in priority_ids:
        if pid in cat_dict:
            return {'cat_id': cat_dict[pid].get('cat_id'), 'cat_name': cat_dict[pid].get('cat_name')}
    
    return {'cat_id': None, 'cat_name': None}

# 各イベントに対してカテゴリを適用
for event in detailed_events:
    picked = pick_priority_category(event.get('event_categories', []))
    event['cat_id'] = picked['cat_id']
    event['cat_name'] = picked['cat_name']

# 必要なカラムのみを抽出して DataFrame 化
event_columns = [
    'event_id', 'event_title', 'event_venue', 'event_date', 'event_status', 
    'event_country', 'event_country_id', 'event_region_name', 'event_region_id',
    'cat_id', 'cat_name'
]

df_target_races = pd.DataFrame(detailed_events)[event_columns]
df_target_programs = json_normalize(all_programs)

print(f"\nSummary:\n- Races: {len(df_target_races)}\n- Total Programs: {len(df_target_programs)}")
df_target_races.to_csv('target_races_detailed.csv', index=False)
display(df_target_races.head())

Processing 10 events...
Fetched 5/10 details
Fetched 10/10 details

Summary:
- Races: 10
- Total Programs: 191


,prog_id,prog_name,event_id,event_title
0,545470,Elite Coaches,163472,2022 World Triathlon Championship Series Leeds
1,545471,NF Medicals,163472,2022 World Triathlon Championship Series Leeds
2,545472,Elite Men,163472,2022 World Triathlon Championship Series Leeds
3,545473,Elite Women,163472,2022 World Triathlon Championship Series Leeds
4,550671,Mixed Relay,163472,2022 World Triathlon Championship Series Leeds


## 3. Step 3: All Relay Results (Team Level)
各 Mixed Relay プログラムの詳細なリザルトをチーム単位で取得し、1つの DataFrame に統合します。

**API Flow:** `/events/{event_id}/programs/{prog_id}/results` -> `data['results']` (List of Teams)
**Data Object:**
- `df_all_relay_results`: チーム単位のリザルト (FK: `prog_id`, `event_id`)

In [3]:
all_team_results = []

# Mixed Relayプログラムのみを抽出
df_mr_programs = df_target_programs[df_target_programs['prog_name'].str.contains('Mixed Relay', case=False, na=False)]

print(f"Fetching results for {len(df_mr_programs)} programs...")
for idx, row in df_mr_programs.iterrows():
    eid, pid = row['event_id'], row['prog_id']
    url = f"{CONFIG['BASE_URL']}/events/{eid}/programs/{pid}/results"
    resp = requests.get(url, headers=HEADERS)
    
    if resp.status_code == 200:
        data = resp.json().get('data', {})
        # 'results' キー配下のチームリストを取得
        team_list = data.get('results', [])
        
        if team_list:
            for team in team_list:
                # リレーション維持のためのキーを注入
                team['prog_id'] = pid
                team['event_id'] = eid
                team['event_title'] = row['event_title']
                all_team_results.append(team)
            print(f"- {row['event_title']}: {len(team_list)} teams fetched")
        else:
            print(f"- {row['event_title']}: No results found")
    else:
        print(f"- FAILED: {row['event_title']} (HTTP {resp.status_code})")
        
    time.sleep(CONFIG['POLITE_DELAY'])

# チーム単位のリザルトを正規化
if all_team_results:
    df_all_relay_results = json_normalize(all_team_results)
    print(f"\nConsolidated {len(df_all_relay_results)} team results across all races.")
    df_all_relay_results.to_csv('all_relay_results.csv', index=False)
    display(df_all_relay_results)
else:
    print("No team results found.")

Fetching results for 10 programs...
- 2022 World Triathlon Championship Series Leeds: 20 teams fetched
- 2022 World Triathlon Sprint and Relay Championships Montreal: 17 teams fetched
- 2022 World Triathlon Championship Series Hamburg: 20 teams fetched
- 2023 Americas Triathlon Cup and South American Championships Lima: 11 teams fetched
- 2023 World Triathlon Championship Series Montreal: No results found
- 2023 World Triathlon Sprint and Relay Championships Hamburg: 22 teams fetched
- 2023 World Triathlon Championship Series Sunderland: 19 teams fetched
- 2023 World Triathlon Mixed Relay Series Paris: 21 teams fetched
- 2024 World Triathlon Championship Series Abu Dhabi: No results found
- 2024 World Triathlon Mixed Relay Olympic Qualification Event Huatulco: 9 teams fetched

Consolidated 139 team results across all races.


,team_id,team_title,team_country_id,team_country_name,team_noc,team_country_isoa2,team_flag,team_flag_circle,profile_edit_date,team_profile_image,...,team_api_listing,team_members,splits,result_id,position,total_time,start_num,prog_id,event_id,event_title
0,126776,Team I Germany,170,Germany,GER,DE,https://cms.triathlon.org/assets/55d8ab45-2131...,https://cms.triathlon.org/assets/b385bbd2-994b...,None,None,...,https://api.triathlon.org/v1/athletes/126776,"[{'splits': ['00:03:53', '00:01:02', '00:09:51...","[00:20:53, 00:23:26, 00:21:04, 00:22:40]",1058460,1,01:28:00,1,550671,163472,2022 World Triathlon Championship Series Leeds
1,126779,Team I Great Britain,292,Great Britain,GBR,GB,https://cms.triathlon.org/assets/d05999e7-1f37...,https://cms.triathlon.org/assets/b75caa96-a00b...,None,None,...,https://api.triathlon.org/v1/athletes/126779,"[{'splits': ['00:03:50', '00:01:03', '00:09:51...","[00:21:03, 00:23:12, 00:21:51, 00:22:10]",1058461,2,01:28:14,4,550671,163472,2022 World Triathlon Championship Series Leeds
2,126787,Team I France,166,France,FRA,FR,https://cms.triathlon.org/assets/3e062fbf-4395...,https://cms.triathlon.org/assets/0730c9c1-0827...,None,None,...,https://api.triathlon.org/v1/athletes/126787,"[{'splits': ['00:03:50', '00:01:05', '00:09:49...","[00:20:55, 00:23:45, 00:20:45, 00:22:56]",1058462,3,01:28:20,7,550671,163472,2022 World Triathlon Championship Series Leeds
3,126801,Team I Italy,190,Italy,ITA,IT,https://cms.triathlon.org/assets/9b100a17-b57e...,https://cms.triathlon.org/assets/95f6dee9-b0e4...,None,None,...,https://api.triathlon.org/v1/athletes/126801,"[{'splits': ['00:03:43', '00:01:06', '00:09:56...","[00:21:05, 00:23:37, 00:21:19, 00:22:33]",1058463,4,01:28:32,2,550671,163472,2022 World Triathlon Championship Series Leeds
4,126799,Team I Belgium,119,Belgium,BEL,BE,https://cms.triathlon.org/assets/a3cfa31f-22a9...,https://cms.triathlon.org/assets/e587d549-c38f...,None,None,...,https://api.triathlon.org/v1/athletes/126799,"[{'splits': ['00:03:55', '00:01:04', '00:09:47...","[00:21:11, 00:23:10, 00:21:00, 00:23:21]",1058464,5,01:28:40,12,550671,163472,2022 World Triathlon Championship Series Leeds
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
134,126789,Team I Austria,112,Austria,AUT,AT,https://cms.triathlon.org/assets/4b88919a-b8e3...,https://cms.triathlon.org/assets/79a68057-b219...,None,None,...,https://api.triathlon.org/v1/athletes/126789,"[{'splits': ['00:04:20', '00:00:53', '00:08:45...","[00:19:05, 00:21:34, 00:19:12, 00:22:44]",1155940,5,01:22:38,9,630338,186039,2024 World Triathlon Mixed Relay Olympic Quali...
135,126781,Team I Mexico,219,Mexico,MEX,MX,https://cms.triathlon.org/assets/07d6935b-6ee3...,https://cms.triathlon.org/assets/3942e657-0458...,None,None,...,https://api.triathlon.org/v1/athletes/126781,"[{'splits': ['00:04:13', '00:00:56', '00:08:49...","[00:19:17, 00:21:35, 00:20:08, 00:24:04]",1155942,6,01:25:05,6,630338,186039,2024 World Triathlon Mixed Relay Olympic Quali...
136,126823,Team I Ecuador,157,Ecuador,ECU,EC,https://cms.triathlon.org/assets/9ca19e94-059d...,https://cms.triathlon.org/assets/e8339442-d22d...,None,None,...,https://api.triathlon.org/v1/athletes/126823,"[{'splits': ['00:04:22', '00:00:54', '00:09:02...","[00:19:16, 00:22:31, 00:20:50, 00:00:00]",1155944,LAP,LAP,8,630338,186039,2024 World Triathlon Mixed Relay Olympic Quali...
137,126773,Team I Canada,135,Canada,CAN,CA,https://cms.triathlon.org/assets/3a31b523-e01d...,https://cms.triathlon.org/assets/9c815286-2883...,None,None,...,https://api.triathlon.org/v1/athletes/126773,"[{'splits': ['00:04:12', '00:00:55', '00:08:15...","[00:18:06, 00:21:11, 00:19:16, 00:24:43]",1155941,DSQ,DSQ,5,630338,186039,2024 World Triathlon Mixed Relay Olympic Quali...


## 4. Step 4: Individual Athlete Results (Athlete Level)
チームリザルトを選手単位に展開し、分析用のフラットな DataFrame を作成します。

**Data Object:**
- `df_individual_results`: 選手単位のリザルト (FK: `team_id`, `prog_id`, `event_id`)

In [4]:
# 1. team_members カラムを展開 (Explode)
df_exploded = df_all_relay_results.explode('team_members').reset_index(drop=True)

# 2. 展開された辞書から必要な選手情報を抽出
# 必要なカラム: team_order, athlete_id, athlete_title, athlete_gender
df_athlete_details = pd.json_normalize(df_exploded['team_members'])

# 3. 元のチーム情報と、抽出した選手情報を結合
df_individual_results = pd.concat([
    df_exploded.drop(columns=['team_members']),
    df_athlete_details[['team_order', 'athlete_id', 'athlete_title', 'athlete_gender']]
], axis=1)

# 4. イベントカテゴリ情報をマージ (cat_id, cat_name)
df_individual_results = df_individual_results.merge(
    df_target_races[['event_id', 'cat_id', 'cat_name']],
    on='event_id',
    how='left'
)

print(f"Generated {len(df_individual_results)} individual athlete records with category info.")
df_individual_results.to_csv('mr_results_consolidated.csv', index=False)
display(df_individual_results.head())

Generated 556 individual athlete records from 139 teams.


,team_id,team_title,team_country_id,team_country_name,team_noc,team_country_isoa2,team_flag,team_flag_circle,profile_edit_date,team_profile_image,...,position,total_time,start_num,prog_id,event_id,event_title,team_order,athlete_id,athlete_title,athlete_gender
0,126776,Team I Germany,170,Germany,GER,DE,https://cms.triathlon.org/assets/55d8ab45-2131...,https://cms.triathlon.org/assets/b385bbd2-994b...,None,None,...,1,01:28:00,1,550671,163472,2022 World Triathlon Championship Series Leeds,1,73078,Lasse Nygaard Priester,male
1,126776,Team I Germany,170,Germany,GER,DE,https://cms.triathlon.org/assets/55d8ab45-2131...,https://cms.triathlon.org/assets/b385bbd2-994b...,None,None,...,1,01:28:00,1,550671,163472,2022 World Triathlon Championship Series Leeds,2,84871,Anabel Knoll,female
2,126776,Team I Germany,170,Germany,GER,DE,https://cms.triathlon.org/assets/55d8ab45-2131...,https://cms.triathlon.org/assets/b385bbd2-994b...,None,None,...,1,01:28:00,1,550671,163472,2022 World Triathlon Championship Series Leeds,3,70410,Lasse Lührs,male
3,126776,Team I Germany,170,Germany,GER,DE,https://cms.triathlon.org/assets/55d8ab45-2131...,https://cms.triathlon.org/assets/b385bbd2-994b...,None,None,...,1,01:28:00,1,550671,163472,2022 World Triathlon Championship Series Leeds,4,70413,Laura Lindemann,female
4,126779,Team I Great Britain,292,Great Britain,GBR,GB,https://cms.triathlon.org/assets/d05999e7-1f37...,https://cms.triathlon.org/assets/b75caa96-a00b...,None,None,...,2,01:28:14,4,550671,163472,2022 World Triathlon Championship Series Leeds,1,18719,Thomas Bishop,male


In [5]:
selected_columns = [
    'team_id', 'team_title', 'team_country_id', 'team_country_name', 'team_noc',
    'result_id', 'position', 'total_time', 'start_num',
    'prog_id', 'event_id', 'event_title',
    'team_order', 'athlete_id', 'athlete_title', 'athlete_gender'
]

df_individual_results = df_individual_results[selected_columns].copy()
df_individual_results.to_csv('mr_results_consolidated_selected_columns.csv', index=False)
display(df_individual_results.head())

,team_id,team_title,team_country_id,team_country_name,team_noc,result_id,position,total_time,start_num,prog_id,event_id,event_title,team_order,athlete_id,athlete_title,athlete_gender
0,126776,Team I Germany,170,Germany,GER,1058460,1,01:28:00,1,550671,163472,2022 World Triathlon Championship Series Leeds,1,73078,Lasse Nygaard Priester,male
1,126776,Team I Germany,170,Germany,GER,1058460,1,01:28:00,1,550671,163472,2022 World Triathlon Championship Series Leeds,2,84871,Anabel Knoll,female
2,126776,Team I Germany,170,Germany,GER,1058460,1,01:28:00,1,550671,163472,2022 World Triathlon Championship Series Leeds,3,70410,Lasse Lührs,male
3,126776,Team I Germany,170,Germany,GER,1058460,1,01:28:00,1,550671,163472,2022 World Triathlon Championship Series Leeds,4,70413,Laura Lindemann,female
4,126779,Team I Great Britain,292,Great Britain,GBR,1058461,2,01:28:14,4,550671,163472,2022 World Triathlon Championship Series Leeds,1,18719,Thomas Bishop,male


In [6]:
# レース粒度
selected_columns_res = [
    'team_id',
    'team_title',
    'team_country_id',
    'team_country_name',
    'team_noc',
    'result_id',
    'position',
    'total_time',
    'start_num',
    'prog_id',
    'event_id',
    'event_title'
]

df_all_relay_results = df_all_relay_results[selected_columns_res].copy()
display(df_individual_results.head())

,team_id,team_title,team_country_id,team_country_name,team_noc,result_id,position,total_time,start_num,prog_id,event_id,event_title,team_order,athlete_id,athlete_title,athlete_gender
0,126776,Team I Germany,170,Germany,GER,1058460,1,01:28:00,1,550671,163472,2022 World Triathlon Championship Series Leeds,1,73078,Lasse Nygaard Priester,male
1,126776,Team I Germany,170,Germany,GER,1058460,1,01:28:00,1,550671,163472,2022 World Triathlon Championship Series Leeds,2,84871,Anabel Knoll,female
2,126776,Team I Germany,170,Germany,GER,1058460,1,01:28:00,1,550671,163472,2022 World Triathlon Championship Series Leeds,3,70410,Lasse Lührs,male
3,126776,Team I Germany,170,Germany,GER,1058460,1,01:28:00,1,550671,163472,2022 World Triathlon Championship Series Leeds,4,70413,Laura Lindemann,female
4,126779,Team I Great Britain,292,Great Britain,GBR,1058461,2,01:28:14,4,550671,163472,2022 World Triathlon Championship Series Leeds,1,18719,Thomas Bishop,male
